# LoRA (Low-Rank Adaptation) Implementation from Scratch

## Introduction
**Low-Rank Adaptation (LoRA)** is a parameter-efficient fine-tuning technique for Large Language Models (LLMs) and neural networks. Instead of fine-tuning all the weights of a standard dense layer, LoRA freezes the pre-trained weights ($W_0$) and injects a pair of trainable low-rank decomposition matrices ($A$ and $B$).

The update is represented as:
$$
W = W_0 + \Delta W = W_0 + B \cdot A
$$
Where:
- $W_0 \in \mathbb{R}^{d \times k}$ is the frozen pre-trained weight matrix.
- $B \in \mathbb{R}^{d \times r}$ and $A \in \mathbb{R}^{r \times k}$ are trainable matrices with rank $r \ll \min(d, k)$.

This significantly reduces the number of trainable parameters while maintaining model performance.

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt

print("TensorFlow version:", tf.__version__)

## 1. Data Generation
We generate synthetic data representing house prices based on size and location. 
The task is a simple regression problem: $Price = 50 \times Size + 30 \times Location + Noise$.

In [ ]:
# Generate synthetic data for house prices
np.random.seed(42)
num_samples = 500

# Features: house size (sq ft) and location index
X = np.random.rand(num_samples, 2) * 100

# True price equation: Price = 50 * Size + 30 * Location + Noise
y = 50 * X[:, 0] + 30 * X[:, 1] + np.random.randn(num_samples) * 10
y = y.reshape(-1, 1)

# Split into training and test sets
train_size = int(0.8 * num_samples)
X_train, y_train = X[:train_size], y[:train_size]
X_test, y_test = X[train_size:], y[train_size:]

print(f"Training Samples: {X_train.shape}, Test Samples: {X_test.shape}")

## 2. Implementing the LoRA Layer

Below is the custom Keras layer creating the LoRA mechanism.

**Key components:**
1.  **`self.W`**: The frozen weight matrix (simulating a pre-trained layer). In a real scenario, this would be initialized with weights from a pretrained model.
2.  **`self.A` & `self.B`**: The trainable low-rank matrices.
3.  **Forward Pass**: Compute `inputs @ W` (frozen path) + `inputs @ A @ B * alpha` (trainable path).

In [ ]:
class LoRALayer(tf.keras.layers.Layer):
    def __init__(self, input_dim, output_dim, rank=2, alpha=1.0, **kwargs):
        super(LoRALayer, self).__init__(**kwargs)
        self.input_dim = input_dim
        self.output_dim = output_dim
        self.rank = rank
        self.alpha = alpha  # Scaling factor

    def build(self, input_shape):
        # Main frozen weight matrix (simulating pre-trained weights)
        # trainable=False ensures these weights are NOT updated during backprop
        self.W = self.add_weight(
            shape=(self.input_dim, self.output_dim),
            initializer="random_normal",
            trainable=False,
            name="frozen_weights"
        )

        # LoRA trainable low-rank matrices
        # A: input_dim -> rank
        self.A = self.add_weight(
            shape=(self.input_dim, self.rank),
            initializer="random_normal",
            trainable=True,
            name="lora_A"
        )
        # B: rank -> output_dim
        self.B = self.add_weight(
            shape=(self.rank, self.output_dim),
            initializer="random_normal", # Often initialized to zeros in practice to start with 0 effect
            trainable=True,
            name="lora_B"
        )
        super(LoRALayer, self).build(input_shape)

    def call(self, inputs):
        # Frozen Path: Standard dense layer calculation with frozen weights
        frozen_out = tf.matmul(inputs, self.W)
        
        # LoRA Path: Low-rank adaptation
        # x @ A @ B
        lora_out = tf.matmul(inputs, self.A)
        lora_out = tf.matmul(lora_out, self.B) * self.alpha
        
        # Combine paths
        return frozen_out + lora_out
        
    def get_config(self):
        config = super(LoRALayer, self).get_config()
        config.update({
            "input_dim": self.input_dim,
            "output_dim": self.output_dim,
            "rank": self.rank,
            "alpha": self.alpha
        })
        return config

## 3. Building the Model

We create a simple regression model. The first hidden layer is our `LoRALayer`. 
Notice the difference in trainable parameters compared to the total parameters.

In [ ]:
# Model parameters
input_dim = X_train.shape[1]
hidden_dim = 16
rank = 2
alpha = 1.0

# Define the model using the Functional API
inputs = keras.Input(shape=(input_dim,))

# Apply LoRA Layer: Input (2) -> Hidden (16)
x = LoRALayer(input_dim, hidden_dim, rank=rank, alpha=alpha)(inputs)
x = layers.ReLU()(x)

# Final Output Layer: Hidden (16) -> Output (1)
outputs = layers.Dense(1)(x)

model = keras.Model(inputs, outputs)

# Compile the model
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.01),
    loss="mse"
)

model.summary()

## 4. Parameter Efficiency Analysis
Let's see how many parameters are actually being trained.

- **LoRA Layer Frozen Weights `W`**: $2 \times 16 = 32$ (Not trained)
- **LoRA Layer Trainable Weights `A`**: $2 \times 2 = 4$
- **LoRA Layer Trainable Weights `B`**: $2 \times 16 = 32$
- **Total Trainable in LoRA**: $36$
- **Ratio**: $36$ trainable vs $32$ frozen (in this small 2D example the savings aren't massive, but in LLMs with dimension 4096+, the savings are huge -- e.g., $4096 \times 4096 \approx 16M$ vs $2 \times 4096 \times 8 \approx 65k$).

## 5. Training

In [ ]:
history = model.fit(
    X_train, 
    y_train, 
    epochs=100, 
    batch_size=16, 
    validation_data=(X_test, y_test), 
    verbose=0 # Set verbose to 1 to see per-epoch logs
)

print("Training Complete.")

## 6. Evaluation and Visualization

In [ ]:
# Plot the loss curves
plt.figure(figsize=(10, 5))
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Model Loss (MSE) over Epochs')
plt.ylabel('Loss (MSE)')
plt.xlabel('Epoch')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Predict on test data
y_pred = model.predict(X_test)

# Visualize Predictions vs True Values
plt.figure(figsize=(10, 5))
plt.scatter(y_test, y_pred, alpha=0.7)
plt.plot([min(y_test), max(y_test)], [min(y_test), max(y_test)], color='red', linestyle='--') # Perfect prediction line
plt.title('Actual vs Predicted Prices')
plt.xlabel('Actual Price')
plt.ylabel('Predicted Price')
plt.grid(True)
plt.show()

In [ ]:
# Example Prediction
sample_idx = 0
print(f"Sample Input (Size, Location): {X_test[sample_idx]}")
print(f"True Price: {y_test[sample_idx][0]:.2f}")
print(f"Predicted Price: {y_pred[sample_idx][0]:.2f}")

## Conclusion

We successfully implemented a custom **LoRA Layer** from scratch in TensorFlow/Keras. 

1.  **Frozen Weights ($W$)**: We demonstrated how to keep the main weights non-trainable.
2.  **Low-Rank Adaptation ($A, B$)**: We injected trainable low-rank matrices to adapt the model.
3.  **Result**: The model successfully learned the regression task using the LoRA path to correct/adapt the random frozen initialization.

This concept is the core mechanism behind efficient fine-tuning of massive models like LLaMA, GPT-4, and Stable Diffusion.